In [1]:
import parcels
import math
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import scipy

import trajan
import cartopy.crs as ccrs
import cartopy.feature
from datetime import datetime, timedelta
import calculate_distance as c_d
import calculate_2DHistogram_method01 as c_H
import grid_initialization as gi
import matplotlib.colors as mcolors
import cmocean.cm as cmo
from matplotlib import colormaps as mcolormaps

In [ ]:
DSS1 = xr.open_zarr("/storage/shared/oceanparcels/output_data/data_Elena/SATELLITE_OUT/SIM4_LA_2024_07_01_kN=0.001.zarr").dropna(dim='obs', how='all')

from IPython.display import HTML
from matplotlib.animation import FuncAnimation, FFMpegWriter

vmin = DSS1["biomass_SF3"].values.min()
vmax = DSS1["biomass_SF3"].values.max()

fig = plt.figure(figsize = (9,5), constrained_layout=True, dpi = 250)
ax = plt.axes(projection=ccrs.PlateCarree())

ax.gridlines(draw_labels=['left','bottom'], zorder=1, alpha=0.3, linestyle='--')
ax.add_feature(cartopy.feature.COASTLINE.with_scale('10m'),zorder=2)
ax.add_feature(cartopy.feature.LAND.with_scale('10m'), zorder=3)

ax.set_extent([-80,-15,-1,26])

#Showing only every 2th output (for speed in creating the animation)
timerange = np.unique(DSS1["time"].values)[::2]

#Indices of the data where time = 0
time_id = np.where(DSS1["time"] == timerange[0])

#sc = ax.scatter(DSS1["lon"].values[time_id], DSS1["lat"].values[time_id], s = 0.3, c = 'seagreen')

scat = ax.scatter(
    DSS1.lon.values[time_id],
    DSS1.lat.values[time_id],
    c=DSS1.biomass_SF3.values[time_id].flatten(),
    cmap=cmo.algae,
    s=0.3,
    vmin=vmin, vmax=vmax,
    transform=ccrs.PlateCarree(), zorder=4
)

# t = str(timerange[0].astype("timedelta64[h]")) 
# title = ax.set_title(f"Sargassum particles at t = {t}")
# print(t)

date_str = np.datetime_as_string(timerange[0], unit='D')
title = ax.set_title(f"Trajectories & relative biomass of Sargassum particles: {date_str}")

def animate(i):
    # t = (timerange[i] - timerange[0]).astype('timedelta64[s]').astype(float) / ( 24 * 60 * 60)
    # #timerange = (DSS1["time"].values)[::2].astype("datetime64[D]")
    # #date = timerange[i]
    # title.set_text(f"Particles at {t:.1f}")
    date_str = np.datetime_as_string(timerange[i], unit='D')
    title.set_text(f"Trajectories & relative biomass of Sargassum particles: {date_str}")

    time_id = np.where(DSS1["time"] == timerange[i])
    scat.set_offsets(np.c_[DSS1["lon"].values[time_id], DSS1["lat"].values[time_id]])
    scat.set_array(DSS1["biomass_SF3"].values[time_id].flatten())

cbar = plt.colorbar(scat, ax=ax, orientation='vertical', label='Relative biomass', shrink=0.5)
anim = FuncAnimation(fig, animate, frames=len(timerange), interval=25)

writervideo = FFMpegWriter(fps=25)
anim.save('_Anim_sat_01072024_pres.mp4', writer=writervideo)
plt.close()

In [ ]:
DS = xr.open_zarr("/storage/shared/oceanparcels/output_data/data_Elena/FINAL/SIM1_LG_EU_2024_07.zarr").dropna(dim='obs', how='all')
DS

from IPython.display import HTML
from matplotlib.animation import FuncAnimation, FFMpegWriter

vmin = DS["limitation"].values.min()
vmax = DS["limitation"].values.max()
vmin = 0
vmax = 1

fig = plt.figure(figsize = (9,5), constrained_layout=True, dpi = 250)
ax = plt.axes(projection=ccrs.PlateCarree())

ax.gridlines(draw_labels=['left','bottom'], zorder=6, alpha=0.3, linestyle='--')
ax.add_feature(cartopy.feature.COASTLINE.with_scale('10m'),zorder=8)
ax.add_feature(cartopy.feature.LAND.with_scale('10m'), zorder=7)

ax.set_extent([-75, -10, -3, 20])

#Showing only every 2th output (for speed in creating the animation)
timerange = np.unique(DS["time"].values)[::2]

#Indices of the data where time = 0
time_id = np.where(DS["time"] == timerange[0])

#sc = ax.scatter(DSS1["lon"].values[time_id], DSS1["lat"].values[time_id], s = 0.3, c = 'seagreen')

scat = ax.scatter(
    DS.lon.values[time_id],
    DS.lat.values[time_id],
    c=DS.limitation.values[time_id].flatten(),
    cmap=cmo.speed,
    s=1,
    vmin=vmin, vmax=vmax,
    transform=ccrs.PlateCarree(), zorder=4
)

# t = str(timerange[0].astype("timedelta64[h]")) 
# title = ax.set_title(f"Sargassum particles at t = {t}")
# print(t)

date_str = np.datetime_as_string(timerange[0], unit='D')
title = ax.set_title(f"Total growth factor: {date_str}")

def animate(i):
    # t = (timerange[i] - timerange[0]).astype('timedelta64[s]').astype(float) / ( 24 * 60 * 60)
    # #timerange = (DSS1["time"].values)[::2].astype("datetime64[D]")
    # #date = timerange[i]
    # title.set_text(f"Particles at {t:.1f}")
    date_str = np.datetime_as_string(timerange[i], unit='D')
    title.set_text(f"Total growth factor: {date_str}")

    time_id = np.where(DS["time"] == timerange[i])
    scat.set_offsets(np.c_[DS["lon"].values[time_id], DS["lat"].values[time_id]])
    scat.set_array(DS["limitation"].values[time_id].flatten())

cbar = plt.colorbar(scat, ax=ax, orientation='vertical', label='Total growth factor', shrink=0.5)
anim = FuncAnimation(fig, animate, frames=len(timerange), interval=25)

writervideo = FFMpegWriter(fps=25)
anim.save('_Anim_growthfactor_pres2.mp4', writer=writervideo)
plt.close()